<a href="https://colab.research.google.com/github/KrathK9722/Analise-de-Dados-Python-Projeto-Avaliativo-M1W07/blob/main/Analise_Google_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1 - Importação das bibliotecas:**

In [741]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

---
## 2 - **Carregamento dos dados**:

In [742]:
id_arquivo = "1iNVCXfu2iFVVHlffBz-fky7U28NhhNv9"

url = f"https://drive.google.com/uc?export=download&id={id_arquivo}"

dados_originais = pd.read_csv(url,sep=";", encoding="latin1")

Sepação feita por ";" porque o padrão estava como "," o que fazia com que o CSV fosse importado com somente uma coluna.

# **3 - Visualização dos dados brutos:**

In [743]:
dados_originais.head(30)

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,NaN,NaN,NaN,NaN
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,NaN,NaN,NaN,NaN
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,NaN,NaN,NaN,NaN
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO,NaN,NaN,NaN,NaN
5,01/02/2019,1000,534,M,4,1,C,187,HIGIENE,HASTES FLEXIVEIS,NaN,NaN,NaN,NaN
6,01/02/2019,1000,534,M,4,1,C,163,ALIMENTOS,MORTADELA,NaN,NaN,NaN,NaN
7,01/02/2019,1000,534,M,4,1,C,11,ALIMENTOS,AZEITE,NaN,NaN,NaN,NaN
8,01/02/2019,1000,534,M,4,1,C,95,LIMPEZA,AMACIANTE,NaN,NaN,NaN,NaN
9,01/02/2019,1000,534,M,4,1,C,198,BEBIDAS,ENERGETICO,NaN,NaN,NaN,NaN


# **3.1 - Quantide de Colunas e linhas**

In [744]:
linhas, colunas = dados_originais.shape

print(f"O número de colunas da tabela é {colunas} e o número total de linhas/registros é {linhas}.")

O número de colunas da tabela é 14 e o número total de linhas/registros é 830000.


-----

### O QUE É CADA COLUNA?

1. DATA: Data da compra;
2. CO_ID: Identificação do número de compra (número da nota fiscal);
3. CL_ID: Identificação do cliente (número do cliente);
4. CL_GENERO: Sexo biológico informado pelo cliente;
5. CL_EC: Estado civil do cliente:
    
    1: Casado ou união estával;
    
    2: Divorciado;
    
    3: Separado;
    
    4: Solteiro;
    
    5: Viúvo.
6. CL_FHL: Número de filhos do cliente;
7. CL_SEG: Segmentação econômica do cliente (classe A, B ou C);
8. PR_ID: Código do produto (SKU) adquirido;
9. PR_CAT: Categoria do produto adquirido;
10. PR_NOME: Nome do produto adquirido.


Demais colunas não contem dados e devem ser removidas no processo de limpeza.

***`Informações retiradas do documento de analise exploratoria da base de varejo.csv`***

-----

# **3.2 - Tipos de dados e outras informações:**

In [745]:
dados_originais.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 830000 entries, 0 to 829999
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   DATA         830000 non-null  object 
 1   CO_ID        830000 non-null  int64  
 2   CL_ID        830000 non-null  int64  
 3   CL_GENERO    830000 non-null  object 
 4   CL_EC        830000 non-null  int64  
 5   CL_FHL       830000 non-null  int64  
 6   CL_SEG       830000 non-null  object 
 7   PR_ID        830000 non-null  int64  
 8   PR_CAT       830000 non-null  object 
 9   PR_NOME      830000 non-null  object 
 10  Unnamed: 10  0 non-null       float64
 11  Unnamed: 11  0 non-null       float64
 12  Unnamed: 12  0 non-null       float64
 13  Unnamed: 13  0 non-null       float64
dtypes: float64(4), int64(5), object(5)
memory usage: 88.7+ MB


#**4 - Limpeza e Validação dos dados**

Copia do banco de dados original para garantir que os dados originais não sejam modificados e possam ser acessados assim como vieram, permitindo que a tabela seja modificada na nova copia.

In [746]:
dados_modificados = pd.read_csv(url,sep=";", encoding="latin1", na_values=["#N/D"])

Modificação na nova tabela para que o pandas entenda que #N/D é nulo e some ao procurar pelos valores nulos.

### **4.1 - Linhas Duplicadas**

In [747]:
duplicados = dados_modificados.duplicated()
print(f"Linhas duplicadas: {duplicados.sum()}")

Linhas duplicadas: 96553


Agora que observados que existem registros duplicados precisamos entender se eles realmente são registros duplicados ou apenas registros de um mesmo produto comprado 2 vezes pela mesma pessoa.

In [748]:
repeticoes_legitimas = dados_modificados.groupby(["CO_ID", "PR_ID"]).size()
print(repeticoes_legitimas[repeticoes_legitimas > 1].head(10))

CO_ID  PR_ID
1000   4        2
       11       2
       13       2
       69       2
       218      2
       225      2
1078   21       2
       23       2
       25       2
       36       2
dtype: int64


Aqui visualizamos registros de notas fiscais com o mesmo produto agora precisamos ver se o resto dos dados nesse registro são identicos.

In [749]:
exemplo = dados_modificados[(dados_modificados["CO_ID"] == 1000) & (dados_modificados["PR_ID"] == 4)]
print(exemplo)

          DATA  CO_ID  CL_ID CL_GENERO  CL_EC  CL_FHL CL_SEG  PR_ID  \
3   01/02/2019   1000    534         M      4       1      C      4   
40  01/02/2019   1000    534         M      4       1      C      4   

       PR_CAT  PR_NOME  Unnamed: 10  Unnamed: 11  Unnamed: 12  Unnamed: 13  
3   ALIMENTOS  ABACAXI          NaN          NaN          NaN          NaN  
40  ALIMENTOS  ABACAXI          NaN          NaN          NaN          NaN  


Diversos dados duplicados foram encontrados isso nos trás algumas questões, podemos remover os dados supondo que são erros de registro ou levando em consideração que a tabela não tem a coluna quantidade podemos entender que as linhas duplicadas de um mesmo registro da compra de produto esta relacionado a quantidade de produtos compradas sendo cada linha um poroduto. Como solução resolvi levar em consideração a segunda opção e criar uma coluna de quantidades removendo as linhas duplicadas para facilitar a análise.

Obs: Vi depois no desafio que a segunda opção era realmente a correta a se seguir.

In [750]:
dados_com_quantidade = dados_modificados.groupby(
    ["CO_ID", "CL_ID","CL_GENERO","CL_EC","CL_FHL","CL_SEG", "PR_ID", "PR_CAT", "PR_NOME","DATA"],
    as_index=False,
    dropna=False
).size()

dados_com_quantidade = dados_com_quantidade.rename(columns={
    "DATA":"DATA_BR"
})

# Renomeia a coluna criada pelo .size() pra um nome mais claro
dados_com_quantidade = dados_com_quantidade.rename(columns={"size": "QUANTIDADE"})
coluna_data = dados_com_quantidade.pop("DATA_BR")
dados_com_quantidade["DATA_BR"] = coluna_data

Criação de uma nova tabela com coluna quantidades juntando os itens comprados mais de uma vez pela mesma pessoa em uma só compra na coluna "QUANTIDADE" a partir da função groupby que automaticamente ja junta linhas iguais, nome "DATA" alterado para "DATA_BR" para facilitar o entendimento futuro e separação de data internacional e data brasileira na conversão da data para tipo DATE_TIME.
Também mudei a posição da coluna "DATA_BR" para posição de ultima coluna por preferencia pessoal de organização.

Observação: O comando GroupBy no pandas remove automaticamente as linhas com valores nulos então para garantir que isso não ocorresse sem as devidas verificações decidi deixar o atributo dropna como false para que o pandas não fizesse isso automaticamente, assim como removi as colunas sem nome ao criar a coluna quantidade ja que colunas sem nome são inuteis para nossa análise ainda mais sabendo que todos os seus valores eram vázios.

In [751]:
dados_com_quantidade = dados_com_quantidade.drop_duplicates()

In [752]:
duplicados_limpos = dados_com_quantidade.duplicated()
print(f"Linhas duplicadas: {duplicados_limpos.sum()}")

Linhas duplicadas: 0


In [753]:
dados_com_quantidade

,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,QUANTIDADE,DATA_BR
0,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,2,01/02/2019
1,1000,534,M,4,1,C,11,ALIMENTOS,AZEITE,2,01/02/2019
2,1000,534,M,4,1,C,13,ALIMENTOS,BANANA,2,01/02/2019
3,1000,534,M,4,1,C,23,ALIMENTOS,COGUMELOS,1,01/02/2019
4,1000,534,M,4,1,C,24,ALIMENTOS,COPA SUINA,1,01/02/2019
...,...,...,...,...,...,...,...,...,...,...,...
733442,919822,155,F,2,0,B,212,LIMPEZA,CERA,2,19/08/2022
733443,919822,155,F,2,0,B,214,ALIMENTOS,CEBOLA,1,19/08/2022
733444,919822,155,F,2,0,B,225,ALIMENTOS,ATUM,1,19/08/2022
733445,919822,155,F,2,0,B,227,ALIMENTOS,ARROZ,2,19/08/2022


Feita a limpeza agrupamento e remoção das linhas duplicadas agora seguimos para
a validação de valores nulos.

### **4.2 - Valores Nulos**

In [754]:
nulos = dados_com_quantidade.isna().sum()
print(f"Valores nulos: \n{nulos}")

Valores nulos: 
CO_ID            0
CL_ID            0
CL_GENERO        0
CL_EC            0
CL_FHL           0
CL_SEG           0
PR_ID            0
PR_CAT        3228
PR_NOME       3228
QUANTIDADE       0
DATA_BR          0
dtype: int64


Podemos ver que existem diversas categorias e nome de produto nulos, sabendo disso precisamos descobrir se conseguimos preencher esses valores corretamente ou se é necessários remover eles.

In [755]:
mapa_categoria = dados_com_quantidade.dropna(subset=["PR_CAT"]).drop_duplicates("PR_ID").set_index("PR_ID")["PR_CAT"]

dados_com_quantidade["PR_CAT"] = dados_com_quantidade["PR_CAT"].fillna(dados_com_quantidade["PR_ID"].map(mapa_categoria))

Criação de um dicionário para tentar preencher as categorias e nomes nulos com o valor correto de acordo com o ID do produto procurnado produtos de mesmo ID na tabela para coletar os nomes e categorias referentes ao ID.

In [756]:
# Pega quantos produtos unicos tem categoria única
ids_com_nulo = dados_com_quantidade.loc[dados_com_quantidade["PR_CAT"].isna(), "PR_ID"].unique()
print(f"Quantidade de produtos únicos com categoria nula: {len(ids_com_nulo)}")

# Procura quantos desses produtos tem essa categoria preenchida em outra linha
ids_recuperaveis = dados_com_quantidade.loc[dados_com_quantidade["PR_ID"].isin(ids_com_nulo) & dados_com_quantidade["PR_CAT"].notna(), "PR_ID"].unique()
print(f"Desses, quantos têm categoria em outra linha: {len(ids_recuperaveis)}")

Quantidade de produtos únicos com categoria nula: 1
Desses, quantos têm categoria em outra linha: 0


In [757]:
# Pega quantos produtos únicos têm nome nulo
ids_com_nulo_nome = dados_com_quantidade.loc[dados_com_quantidade["PR_NOME"].isna(), "PR_ID"].unique()
print(f"Quantidade de produtos únicos com nome nulo: {len(ids_com_nulo_nome)}")

# Desses produtos, quantos têm o nome preenchido em outra linha
ids_recuperaveis_nome = dados_com_quantidade.loc[dados_com_quantidade["PR_ID"].isin(ids_com_nulo_nome) & dados_com_quantidade["PR_NOME"].notna(), "PR_ID"].unique()
print(f"Desses, quantos têm nome em outra linha: {len(ids_recuperaveis_nome)}")

Quantidade de produtos únicos com nome nulo: 1
Desses, quantos têm nome em outra linha: 0


Ao fazer essa visualização dos IDS unicos que tem categoria e nomes nulos podemos entender que todas essas categorias nulas vem de um mesmo produto então precisamos descobrir qual é esse produto.

In [758]:
pr_id_desconhecido = dados_com_quantidade.loc[dados_com_quantidade["PR_CAT"].isna(), "PR_ID"].unique()[0]
print(f"PR_ID: {pr_id_desconhecido}")

# Confirma que o nome do produto também está nulo
print(dados_com_quantidade.loc[dados_com_quantidade["PR_ID"] == pr_id_desconhecido, ["PR_ID", "PR_CAT", "PR_NOME"]].head())

PR_ID: 107
     PR_ID PR_CAT PR_NOME
93     107    NaN     NaN
190    107    NaN     NaN
548    107    NaN     NaN
730    107    NaN     NaN
791    107    NaN     NaN


Agora que descobrimos o produto entendemos que esse produto em especifíco esta com algum problema em seus registros que faz com que o seu nome e categoria não estejam informados. Por isso para facilitar visualizações futuras vamos trocar os dados nesses espaços por "Sem Categoria" e por "Sem Nome" para que qualquer um consiga entender ao visualizar os dados.

In [759]:
dados_limpos = dados_com_quantidade.copy()
linhas, colunas = dados_limpos.shape
categorias_corrigidas = False
nomes_corrigidos = False
n_linha=0
for i in range(linhas):
  if pd.isna(dados_limpos.loc[i, "PR_CAT"]) and categorias_corrigidas == False:
    dados_limpos["PR_CAT"] = dados_limpos["PR_CAT"].fillna("Sem Categoria")
    categorias_corrigidas = True
  if pd.isna(dados_limpos.loc[i, "PR_NOME"]) and nomes_corrigidos == False:
    dados_limpos["PR_NOME"] = dados_limpos["PR_NOME"].fillna("Sem Nome")
    nomes_corrigidos = True
  if categorias_corrigidas == True and nomes_corrigidos == True:
    print("Correção concluida")
  else:
    n_linha += 1
    continue
  print(f"Número de linhas válidadas até a conclusão: {n_linha}")
  break

Correção concluida
Número de linhas válidadas até a conclusão: 93


Validação feita para procurar se algum valor em categoria ou em nome é nulo mesmo e após isso trocar todos os valores vazios por um aviso de "Sem Categoria" ou "Sem Nome" com um if final para verificar se as alterações ja foram feitas e cancelar o loop que levaria muito tempo para passar por todos os valores considerando o grande número de registros.

Obs: Essa etapa poderia ter sido feita utilizando apenas "dados_limpos["PR_CAT"] = dados_limpos["PR_CAT"].fillna("Sem Categoria")" e "dados_limpos["PR_NOME"] = dados_limpos["PR_NOME"].fillna("Sem Nome")" no entanto como o documento pede que seja feito o uso de If e Else achei interessante criar um loop que diga quantas linhas precisaram ser validadas para encontrar ao menos um nome nulo e uma categoria nula.

In [760]:
print(dados_limpos.loc[dados_com_quantidade["PR_ID"] == pr_id_desconhecido, ["PR_ID", "PR_CAT", "PR_NOME"]].head())

     PR_ID         PR_CAT   PR_NOME
93     107  Sem Categoria  Sem Nome
190    107  Sem Categoria  Sem Nome
548    107  Sem Categoria  Sem Nome
730    107  Sem Categoria  Sem Nome
791    107  Sem Categoria  Sem Nome


In [761]:
nulos = dados_limpos.isna().sum()
print(f"Valores nulos: \n{nulos}")

Valores nulos: 
CO_ID         0
CL_ID         0
CL_GENERO     0
CL_EC         0
CL_FHL        0
CL_SEG        0
PR_ID         0
PR_CAT        0
PR_NOME       0
QUANTIDADE    0
DATA_BR       0
dtype: int64


Podemos ver que a alteração dos valores foi muito bem sucedida. Após a limpeza e validação de valores nulos e duplicados precisamos validas e converter as Datas de Registro.

### **4.3 - Conversão e Validação das Datas**

In [762]:
dados_data = dados_limpos.copy()

dados_data["DATA_INT"] = pd.to_datetime(dados_data["DATA_BR"], format="%d/%m/%Y", errors="coerce")

In [763]:
dados_data.head(3)

,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,QUANTIDADE,DATA_BR,DATA_INT
0,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,2,01/02/2019,2019-02-01
1,1000,534,M,4,1,C,11,ALIMENTOS,AZEITE,2,01/02/2019,2019-02-01
2,1000,534,M,4,1,C,13,ALIMENTOS,BANANA,2,01/02/2019,2019-02-01


In [764]:
datas_invalidas = dados_data["DATA_INT"].isna().sum()
print(f"Datas inválidas: {datas_invalidas}")

Datas inválidas: 0


Após validar as datas agora e converter elas separando em DATA_BR(STRING) que facilita entendimento de brasileiros e DATA_INT(DATE_TIME) em formato internacional que facilita análises podemos ir para a próxima etapa que é o agrupamento de informações relevantes e separação correta dos dados para análise.

# **6 - Agrupamento de Dados**

### **AGRUPAMENTO DE COMPRAS:**

In [765]:
compras_agrupadas = dados_data.groupby("CO_ID", as_index=False).agg(
    CL_ID=("CL_ID", "first"),           # O cliente é sempre o mesmo então podemos pegar somente o primeiro
    DATA_INT=("DATA_INT", "first"),       # A data da compra também é a mesma então podemos pegar o primeiro
    QTD_TOTAL_ITENS=("QUANTIDADE", "sum"),      # Soma quantidade total de itens comprados
    QTD_PRODUTOS_DIFERENTES=("PR_ID", "nunique"),# Soma quantidade de itens diferentes na compra
    PRODUTOS=("PR_NOME", list),         # Lista o nome de todos os produtos comprados
    DATA_COMPRA=("DATA_INT", "first")
)

compras_agrupadas.head(10).style.hide(axis="index")

CO_ID,CL_ID,DATA_INT,QTD_TOTAL_ITENS,QTD_PRODUTOS_DIFERENTES,PRODUTOS,DATA_COMPRA
1000,534,2019-02-01 00:00:00,52,46,"['ABACAXI', 'AZEITE', 'BANANA', 'COGUMELOS', 'COPA SUINA', 'CORACAO DE FRANGO', 'PATE', 'QUEIJO MUSSARELA', 'SARDINHA', 'UVA', 'REFRIGERANTE COLA', 'REFRIGERANTE GUARANA', 'REFRIGERANTE LARANJA', 'REFRIGERANTE LIMaO', 'REFRIGERANTE OUTROS', 'PAPEL HIGIENICO', 'PRESERVATIVO', 'REPELENTE', 'AMACIANTE', 'REMOVEDOR', 'UVA', 'TIRA MANCHA', 'SABONETE', 'RODO', 'RICOTA', 'REFRIGERANTE GUARANA', 'QUEIJO MUSSARELA', 'PATE', 'PALMITO', 'MORTADELA', 'LIMPADOR MULTIUSO', 'LIMPA VIDROS', 'LENCO UMEDECIDO', 'LATA DE ERVILHA', 'KETCHUP', 'HASTES FLEXIVEIS', 'GRANOLA', 'FRALDA', 'ENERGETICO', 'DETERGENTE', 'COGUMELOS', 'BIFE DE COXAO MOLE', 'BANANA', 'ATUM', 'ARROZ INTEGRAL', 'ALMONDEGA']",2019-02-01 00:00:00
1040,279,2019-02-01 00:00:00,15,15,"['CAFE', 'LEITE CONDENSADO', 'MAMAO PAPAYA', 'OVOS', 'SALGADINHOS DE MILHO', 'ESCOVA DE DENTE', 'ALCOOL', 'AMACIANTE', 'LUSTRA MOVEIS', 'SABAO', 'RACAO UMIDA PARA GATOS', 'QUEIJO MUSSARELA', 'MORDEDOR', 'LEITE', 'CONDICIONADOR']",2019-02-01 00:00:00
1078,290,2019-02-01 00:00:00,82,63,"['CHUPETA', 'ACHOCOLATADO', 'ALHO', 'BANANA', 'BATATA', 'BISCOITO', 'CAFE', 'CENOURA', 'CHA', 'COGUMELOS', 'CORACAO DE FRANGO', 'FILE DE PEIXE', 'LATA DE ERVILHA', 'LATA DE MILHO', 'LIMAO', 'MAMAO', 'OVOS', 'PALMITO', 'QUEIJO MUSSARELA', 'SALGADINHO', 'SNACKS', 'TOMATE', 'FIO DENTAL', 'HIDRATANTE', 'LENCO UMEDECIDO', 'PAPEL HIGIENICO', 'SHAMPOO', 'AGUA SANITARIA', 'DESINFETANTE', 'LIMPA VIDROS', 'LUSTRA MOVEIS', 'ODORIZADOR', 'Sem Nome', 'RODO', 'TIRA MANCHA', 'RACAO SECA PARA CAES', 'RACAO UMIDA PARA CAES', 'TOMATE', 'TIRA MANCHA', 'TINTURAS', 'SARDINHA', 'RICOTA', 'REQUEIJAO', 'REFRIGERANTE COLA', 'RACAO UMIDA PARA GATOS', 'QUEIJO DE CABRA', 'PROTETOR SOLAR', 'PATE', 'PASTA DE DENTE', 'PALMITO', 'ODORIZADOR', 'LUSTRA MOVEIS', 'LINGUICA', 'LIMPADOR PERFUMADO', 'LEITE', 'GEL', 'FRALDA', 'FIXADOR', 'DETERGENTE', 'CHUPETA', 'CENOURA', 'CAFE', 'ATUM']",2019-02-01 00:00:00
1082,323,2019-02-01 00:00:00,71,62,"['ALMONDEGA', 'AZEITE', 'CAFE', 'DANETTE', 'FILE DE PEIXE', 'LATA DE ERVILHA', 'LEITE CONDENSADO', 'MACARRAO INSTANTANEO', 'MAMAO', 'OVOS', 'PAPINHA INFANTIL', 'PATE', 'PRESUNTO COZIDO', 'PRESUNTO COZIDO', 'SALGADINHO', 'REFRIGERANTE LARANJA', 'REFRIGERANTE LIMaO', 'REFRIGERANTE OUTROS', 'FIO DENTAL', 'FRALDA', 'HASTES FLEXIVEIS', 'PAPEL HIGIENICO', 'PASTA DE DENTE', 'ALCOOL', 'AMACIANTE', 'CERA', 'DESENGORDURANTE', 'SABAO', 'SABAO EM PO', 'TIRA MANCHA', 'VASSOURA', 'RACAO SECA PARA GATOS', 'TOMATE', 'TIRA LIMO', 'TINTURAS', 'TALCO', 'SABAO', 'RODO', 'REFRIGERANTE LIMaO', 'REFRIGERANTE LARANJA', 'REFRIGERANTE GUARANA', 'RACAO UMIDA PARA CAES', 'PRESERVATIVO', 'PASTA DE DENTE', 'PAPINHA INFANTIL', 'PALMITO', 'MODELADOR', 'LUSTRA MOVEIS', 'LINGUICA', 'LEITE', 'INSETICIDA', 'HASTES FLEXIVEIS', 'GEL', 'FEIJAO', 'ENERGETICO', 'DESINFETANTE', 'CREME', 'CERA', 'CENOURA', 'BATATA', 'AMACIANTE', 'ALMONDEGA']",2019-02-01 00:00:00
1103,957,2019-02-01 00:00:00,6,6,"['BATATA DOCE', 'DANETTE', 'DOCE', 'PROTETOR SOLAR', 'Sem Nome', 'LATA DE ERVILHA']",2019-02-01 00:00:00
1121,49,2019-02-01 00:00:00,64,54,"['ABACAXI', 'AZEITONA', 'BATATA DOCE', 'BROCOLIS', 'CENOURA', 'COPA SUINA', 'DOCE', 'HAMBUGUER', 'LATA DE MILHO', 'MAMAO PAPAYA', 'PAO DE FORMA', 'PATE', 'UVA', 'ENERGETICO', 'ESCOVA DE DENTE', 'HASTES FLEXIVEIS', 'LENCO UMEDECIDO', 'PROTETOR SOLAR', 'TALCO', 'TINTURAS', 'AGUA SANITARIA', 'BALDE', 'DESINFETANTE', 'DETERGENTE', 'LIMPA VIDROS', 'REMOVEDOR', 'VASSOURA', 'RACAO SECA PARA CAES', 'RACAO UMIDA PARA GATOS', 'TIRA MANCHA', 'TINTURAS', 'SHAMPOO', 'SARDINHA', 'RACAO UMIDA PARA CAES', 'RACAO SECA PARA CAES', 'PROTETOR SOLAR', 'PAO DE FORMA', 'PALMITO', 'OVOS', 'MORDEDOR', 'MAMAO', 'MACARRAO', 'LINGUICA', 'LIMPADOR PERFUMADO', 'LEITE', 'IOGURTE', 'INSETICIDA', 'COPA SUINA', 'COGUMELOS', 'BISCOITO', 'BATATA', 'BALDE', 'AZEITONA', 'ARROZ INTEGRAL']",2019-02-01 00:00:00
1188,906,2019-02-01 00:00:00,78,67,"['CHUPETA', '

Compreensão da RN relacionada a CO_ID fazendo agrupamento dos itens de uma mesma compra para uso dem análises posteriores.

### **AGRUPAMENTO DE CLIENTES:**

In [766]:
clientes = dados_data.groupby("CL_ID",as_index=False).agg(
    GENERO=("CL_GENERO", "first"),
    ESTADO_CIVIL=("CL_EC", "first"),
    NUMERO_FILHOS=("CL_FHL", "first"),
    SEGMENTO_ECONOMICO=("CL_SEG", "first")
)

clientes.head(10).style.hide(axis="index")

CL_ID,GENERO,ESTADO_CIVIL,NUMERO_FILHOS,SEGMENTO_ECONOMICO
1,M,1,0,C
2,M,3,0,B
3,F,4,0,B
4,M,4,0,B
5,F,4,3,C
6,F,2,0,B
7,M,1,3,B
8,F,3,0,B
9,F,2,0,B
10,M,1,0,B


### **AGRUPAMENTO GERAL DOS DADOS COM COMPRAS AGRUPADAS E DADOS DOS CLIENTES:**

In [767]:
clientes = clientes.reset_index()

dados_gerais = pd.merge(
    compras_agrupadas,
    clientes,
    on="CL_ID",
    how="left"
)
dados_gerais.head(10).style.hide(axis="index")

CO_ID,CL_ID,DATA_INT,QTD_TOTAL_ITENS,QTD_PRODUTOS_DIFERENTES,PRODUTOS,DATA_COMPRA,index,GENERO,ESTADO_CIVIL,NUMERO_FILHOS,SEGMENTO_ECONOMICO
1000,534,2019-02-01 00:00:00,52,46,"['ABACAXI', 'AZEITE', 'BANANA', 'COGUMELOS', 'COPA SUINA', 'CORACAO DE FRANGO', 'PATE', 'QUEIJO MUSSARELA', 'SARDINHA', 'UVA', 'REFRIGERANTE COLA', 'REFRIGERANTE GUARANA', 'REFRIGERANTE LARANJA', 'REFRIGERANTE LIMaO', 'REFRIGERANTE OUTROS', 'PAPEL HIGIENICO', 'PRESERVATIVO', 'REPELENTE', 'AMACIANTE', 'REMOVEDOR', 'UVA', 'TIRA MANCHA', 'SABONETE', 'RODO', 'RICOTA', 'REFRIGERANTE GUARANA', 'QUEIJO MUSSARELA', 'PATE', 'PALMITO', 'MORTADELA', 'LIMPADOR MULTIUSO', 'LIMPA VIDROS', 'LENCO UMEDECIDO', 'LATA DE ERVILHA', 'KETCHUP', 'HASTES FLEXIVEIS', 'GRANOLA', 'FRALDA', 'ENERGETICO', 'DETERGENTE', 'COGUMELOS', 'BIFE DE COXAO MOLE', 'BANANA', 'ATUM', 'ARROZ INTEGRAL', 'ALMONDEGA']",2019-02-01 00:00:00,533,M,4,1,C
1040,279,2019-02-01 00:00:00,15,15,"['CAFE', 'LEITE CONDENSADO', 'MAMAO PAPAYA', 'OVOS', 'SALGADINHOS DE MILHO', 'ESCOVA DE DENTE', 'ALCOOL', 'AMACIANTE', 'LUSTRA MOVEIS', 'SABAO', 'RACAO UMIDA PARA GATOS', 'QUEIJO MUSSARELA', 'MORDEDOR', 'LEITE', 'CONDICIONADOR']",2019-02-01 00:00:00,278,M,1,2,B
1078,290,2019-02-01 00:00:00,82,63,"['CHUPETA', 'ACHOCOLATADO', 'ALHO', 'BANANA', 'BATATA', 'BISCOITO', 'CAFE', 'CENOURA', 'CHA', 'COGUMELOS', 'CORACAO DE FRANGO', 'FILE DE PEIXE', 'LATA DE ERVILHA', 'LATA DE MILHO', 'LIMAO', 'MAMAO', 'OVOS', 'PALMITO', 'QUEIJO MUSSARELA', 'SALGADINHO', 'SNACKS', 'TOMATE', 'FIO DENTAL', 'HIDRATANTE', 'LENCO UMEDECIDO', 'PAPEL HIGIENICO', 'SHAMPOO', 'AGUA SANITARIA', 'DESINFETANTE', 'LIMPA VIDROS', 'LUSTRA MOVEIS', 'ODORIZADOR', 'Sem Nome', 'RODO', 'TIRA MANCHA', 'RACAO SECA PARA CAES', 'RACAO UMIDA PARA CAES', 'TOMATE', 'TIRA MANCHA', 'TINTURAS', 'SARDINHA', 'RICOTA', 'REQUEIJAO', 'REFRIGERANTE COLA', 'RACAO UMIDA PARA GATOS', 'QUEIJO DE CABRA', 'PROTETOR SOLAR', 'PATE', 'PASTA DE DENTE', 'PALMITO', 'ODORIZADOR', 'LUSTRA MOVEIS', 'LINGUICA', 'LIMPADOR PERFUMADO', 'LEITE', 'GEL', 'FRALDA', 'FIXADOR', 'DETERGENTE', 'CHUPETA', 'CENOURA', 'CAFE', 'ATUM']",2019-02-01 00:00:00,289,F,1,0,B
1082,323,2019-02-01 00:00:00,71,62,"['ALMONDEGA', 'AZEITE', 'CAFE', 'DANETTE', 'FILE DE PEIXE', 'LATA DE ERVILHA', 'LEITE CONDENSADO', 'MACARRAO INSTANTANEO', 'MAMAO', 'OVOS', 'PAPINHA INFANTIL', 'PATE', 'PRESUNTO COZIDO', 'PRESUNTO COZIDO', 'SALGADINHO', 'REFRIGERANTE LARANJA', 'REFRIGERANTE LIMaO', 'REFRIGERANTE OUTROS', 'FIO DENTAL', 'FRALDA', 'HASTES FLEXIVEIS', 'PAPEL HIGIENICO', 'PASTA DE DENTE', 'ALCOOL', 'AMACIANTE', 'CERA', 'DESENGORDURANTE', 'SABAO', 'SABAO EM PO', 'TIRA MANCHA', 'VASSOURA', 'RACAO SECA PARA GATOS', 'TOMATE', 'TIRA LIMO', 'TINTURAS', 'TALCO', 'SABAO', 'RODO', 'REFRIGERANTE LIMaO', 'REFRIGERANTE LARANJA', 'REFRIGERANTE GUARANA', 'RACAO UMIDA PARA CAES', 'PRESERVATIVO', 'PASTA DE DENTE', 'PAPINHA INFANTIL', 'PALMITO', 'MODELADOR', 'LUSTRA MOVEIS', 'LINGUICA', 'LEITE', 'INSETICIDA', 'HASTES FLEXIVEIS', 'GEL', 'FEIJAO', 'ENERGETICO', 'DESINFETANTE', 'CREME', 'CERA', 'CENOURA', 'BATATA', 'AMACIANTE', 'ALMONDEGA']",2019-02-01 00:00:00,322,M,1,3,B
1103,957,2019-02-01 00:00:00,6,6,"['BATATA DOCE', 'DANETTE', 'DOCE', 'PROTETOR SOLAR', 'Sem Nome', 'LATA DE ERVILHA']",2019-02-01 00:00:00,956,M,2,3,B
1121,49,2019-02-01 00:00:00,64,54,"['ABACAXI', 'AZEITONA', 'BATATA DOCE', 'BROCOLIS', 'CENOURA', 'COPA SUINA', 'DOCE', 'HAMBUGUER', 'LATA DE MILHO', 'MAMAO PAPAYA', 'PAO DE FORMA', 'PATE', 'UVA', 'ENERGETICO', 'ESCOVA DE DENTE', 'HASTES FLEXIVEIS', 'LENCO UMEDECIDO', 'PROTETOR SOLAR', 'TALCO', 'TINTURAS', 'AGUA SANITARIA', 'BALDE', 'DESINFETANTE', 'DETERGENTE', 'LIMPA VIDROS', 'REMOVEDOR', 'VASSOURA', 'RACAO SECA PARA CAES', 'RACAO UMIDA PARA GATOS', 'TIRA MANCHA', 'TINTURAS', 'SHAMPOO', 'SARDINHA', 'RACAO UMIDA PARA CAES', 'RACAO SECA PARA CAES', 'PROTETOR SOLAR', 'PAO DE FORMA', 'PALMITO', 'OVOS', 'MORDEDOR', 'MAMAO', 'MACARRAO', 'LINGUICA', 'LIMPADOR PERFUMADO', 'LEITE', 'IOGURTE', 'INSETICIDA', 'COPA SUINA', 'COGUMELOS', 'BISCOITO', 

Dados utéis para visualizações gerais como compras por segmento, genero, estado civil e cliente, assim como dados mais detalhados como quantidade de itens totais comprados ou itens diferentes.

### **AGRUPAMENTO DE DADOS POR SEGMENTO:**

In [768]:
resumo_segmento = dados_gerais.groupby("SEGMENTO_ECONOMICO", as_index=False).agg(
    NUMERO_DE_CLIENTES=("CL_ID", "nunique"),
    QUANTIDADE_COMPRAS=("CO_ID", "nunique"),
    QUANTIDADE_TOTAL_DE_ITENS=("QTD_TOTAL_ITENS", "sum")
)

# Já que produtos foi colocado como list e Produtos unicos não pode ser apenas somado precisamos separar de outra forma
produtos_unicos_segmento = dados_gerais.explode("PRODUTOS").groupby(
    "SEGMENTO_ECONOMICO", as_index=False
)["PRODUTOS"].nunique().rename(columns={"PRODUTOS": "QUANTIDADE_DE_ITENS_UNICOS"})

# O Número de filhos precisa ser calculado por cliente único, para que não conte o mesmo cliente uma vez por compra que ele fez
clientes_unicos = dados_gerais.drop_duplicates(subset="CL_ID")
filhos_segmento = clientes_unicos.groupby("SEGMENTO_ECONOMICO", as_index=False)["NUMERO_FILHOS"].sum()

# União dos produtos únicos com os resumo de segmento e número de filhos
dados_segmento = pd.merge(resumo_segmento, produtos_unicos_segmento, on="SEGMENTO_ECONOMICO")
dados_segmento = pd.merge(dados_segmento, filhos_segmento, on="SEGMENTO_ECONOMICO")
dados_segmento.style.hide(axis="index")

SEGMENTO_ECONOMICO,NUMERO_DE_CLIENTES,QUANTIDADE_COMPRAS,QUANTIDADE_TOTAL_DE_ITENS,QUANTIDADE_DE_ITENS_UNICOS,NUMERO_FILHOS
A,84,1492,67736,118,89
B,645,11843,530163,118,724
C,271,5136,232101,118,323


Esses dados são extremamente importantes podemos ver informações detalhadas de cada um dos segmentos o que ajuda a entender o cliente alvo/pessoas que mais compram e a importância delas para o varejo.

In [769]:
total_produtos_catalogo = dados_gerais.explode("PRODUTOS")["PRODUTOS"].nunique()
print(total_produtos_catalogo)

118


Verificação da quantidade total de produtos no catalogo para garantir que a informação recebida em QUANTIDADE_DE_ITENS_UNICOS por segmento esta correta.

### **AGRUPAMENTO DE DADOS POR GENERO:**


In [770]:
# Mesmo código utilizado em dados de segmento para agilizar o processo de análise
resumo_genero = dados_gerais.groupby("GENERO", as_index=False).agg(
    NUMERO_DE_CLIENTES=("CL_ID", "nunique"),
    QUANTIDADE_COMPRAS=("CO_ID", "nunique"),
    QUANTIDADE_TOTAL_DE_ITENS=("QTD_TOTAL_ITENS", "sum")
)

produtos_unicos_genero = dados_gerais.explode("PRODUTOS").groupby(
    "GENERO", as_index=False
)["PRODUTOS"].nunique().rename(columns={"PRODUTOS": "QUANTIDADE_DE_ITENS_UNICOS"})

clientes_unicos_genero = dados_gerais.drop_duplicates(subset="CL_ID")
filhos_genero = clientes_unicos_genero.groupby("GENERO", as_index=False)["NUMERO_FILHOS"].sum()

# União dos produtos únicos com o resumo de gênero e número de filhos
dados_genero = pd.merge(resumo_genero, produtos_unicos_genero, on="GENERO")
dados_genero = pd.merge(dados_genero, filhos_genero, on="GENERO")
dados_genero.style.hide(axis="index")

GENERO,NUMERO_DE_CLIENTES,QUANTIDADE_COMPRAS,QUANTIDADE_TOTAL_DE_ITENS,QUANTIDADE_DE_ITENS_UNICOS,NUMERO_FILHOS
F,519,9615,432576,118,561
M,481,8856,397424,118,575


Agrupado da mesma forma que SEGMENTO ECONÔMICO facilita na visualização de dados específicos e entendimento dos clientes.

### **AGRUPAMENTO POR ESTADO CIVIL:**

In [771]:
# Traduzindo estados civis
mapa_estado_civil = {
    1: "Casado ou União Estável",
    2: "Divorciado",
    3: "Separado",
    4: "Solteiro",
    5: "Viúvo"
}
dados_gerais["ESTADO_CIVIL"] = dados_gerais["ESTADO_CIVIL"].map(mapa_estado_civil)

resumo_estado_civil = dados_gerais.groupby("ESTADO_CIVIL", as_index=False).agg(
    NUMERO_DE_CLIENTES=("CL_ID", "nunique"),
    QUANTIDADE_COMPRAS=("CO_ID", "nunique"),
    QUANTIDADE_TOTAL_DE_ITENS=("QTD_TOTAL_ITENS", "sum")
)

produtos_unicos_estado_civil = dados_gerais.explode("PRODUTOS").groupby(
    "ESTADO_CIVIL", as_index=False
)["PRODUTOS"].nunique().rename(columns={"PRODUTOS": "QUANTIDADE_DE_ITENS_UNICOS"})

clientes_unicos_ec = dados_gerais.drop_duplicates(subset="CL_ID")
filhos_estado_civil = clientes_unicos_ec.groupby("ESTADO_CIVIL", as_index=False)["NUMERO_FILHOS"].sum()

dados_estado_civil = pd.merge(resumo_estado_civil, produtos_unicos_estado_civil, on="ESTADO_CIVIL")
dados_estado_civil = pd.merge(dados_estado_civil, filhos_estado_civil, on="ESTADO_CIVIL")
dados_estado_civil.style.hide(axis="index")

ESTADO_CIVIL,NUMERO_DE_CLIENTES,QUANTIDADE_COMPRAS,QUANTIDADE_TOTAL_DE_ITENS,QUANTIDADE_DE_ITENS_UNICOS,NUMERO_FILHOS
Casado ou União Estável,233,4290,194873,118,295
Divorciado,238,4377,194990,118,297
Separado,250,4762,213742,118,299
Solteiro,251,4551,202618,118,203
Viúvo,28,491,23777,118,42


Agrupamento por estado civil, denovo utilizando mesmo método do agrupamento por genero e do agrupamento por segmento mas dessa vez traduzindo os códigos de estado civil para seus respectivos nomes.

### **EVOLUÇÃO DE COMPRAS AO LONGO DO TEMPO (por mês):**

In [772]:
dados_gerais["MES_ANO"] = dados_gerais["DATA_INT"].dt.to_period("M")

resumo_mensal = dados_gerais.groupby("MES_ANO", as_index=False).agg(
    QUANTIDADE_COMPRAS=("CO_ID", "nunique"),
    QUANTIDADE_TOTAL_DE_ITENS=("QTD_TOTAL_ITENS", "sum")
)

resumo_mensal.head(10).style.hide(axis="index")

MES_ANO,QUANTIDADE_COMPRAS,QUANTIDADE_TOTAL_DE_ITENS
2019-01,171,7368
2019-02,402,18336
2019-03,257,11051
2019-04,410,18337
2019-05,711,31811
2019-06,493,22084
2019-07,274,12184
2019-08,420,17900
2019-09,436,19122
2019-10,242,10245


Tabela de evolução da compra ao longo dos meses, muito util para criação de gráficos e compreender oque mudou nas vendar do estabelecimento ou empresa como um todo.

### **PRODUTOS MAIS VENDIDOS (ranking geral):**

In [773]:
produtos_explodido = dados_gerais.explode("PRODUTOS")

top_10_produtos = produtos_explodido.groupby("PRODUTOS", as_index=False).agg(
    QUANTIDADE_VENDIDA=("CO_ID", "count")   # quantas vezes aquele produto apareceu em compras
).sort_values("QUANTIDADE_VENDIDA", ascending=False).head(10)

top_10_produtos.style.hide(axis="index")

PRODUTOS,QUANTIDADE_VENDIDA
PRESUNTO COZIDO,12719
SARDINHA,6610
BANANA,6518
ESCOVA DE DENTE,6518
GEL,6517
PAPINHA INFANTIL,6515
MODELADOR,6505
CERA,6502
CEBOLA,6501
LIMPADOR PERFUMADO,6501


Bom para saber o que mais gira dentro do estoque.

### **TICKET MÉDIO (itens por compra) POR GÊNERO E SEGMENTO JUNTOS:**

In [774]:
resumo_genero_segmento = dados_gerais.groupby(
    ["GENERO", "SEGMENTO_ECONOMICO"], as_index=False
).agg(
    QUANTIDADE_COMPRAS=("CO_ID", "nunique"),
    QUANTIDADE_TOTAL_DE_ITENS=("QTD_TOTAL_ITENS", "sum")
)

# Calcula o ticket médio (itens por compra) como coluna nova
resumo_genero_segmento["MEDIA_ITENS_POR_COMPRA"] = (
    resumo_genero_segmento["QUANTIDADE_TOTAL_DE_ITENS"] / resumo_genero_segmento["QUANTIDADE_COMPRAS"]
).round(2)

resumo_genero_segmento.style.hide(axis="index")

GENERO,SEGMENTO_ECONOMICO,QUANTIDADE_COMPRAS,QUANTIDADE_TOTAL_DE_ITENS,MEDIA_ITENS_POR_COMPRA
F,A,700,32875,46.960000
F,B,5946,264308,44.450000
F,C,2969,135393,45.600000
M,A,792,34861,44.020000
M,B,5897,265855,45.080000
M,C,2167,96708,44.630000


Ótimo para entender quanto cada tipo de cliente normalmente compra. Por essa tabela, vemos que o ticket
médio entre os grupos é bastante parecido, ainda assim o Segmento Econômico A do gênero Feminino se
destaca com o maior ticket médio, chegando a quase 47 itens por compra. Como a Classe A representa a
faixa de maior renda entre as três (segundo a segmentação econômica da base), isso sugere que clientes
de maior poder aquisitivo tendem a comprar mais itens por vez. Por outro lado, na mesma tabela vemos que
a quantidade de compras efetuadas pelo Segmento A é a menor entre os três grupos, ou seja, esse
segmento compra com menos frequência, porém em maior volume por compra, um padrão coerente com um
perfil de consumidor de maior renda que concentra suas compras em vez de fracioná-las.

Agora que agrupamos alguns valores importantes podemos seguir e visualizar as informações estatísticas da coluna de número de filhos.

# **7 - Estatísticas**

In [775]:
# Estatísticas descritivas da coluna NUMERO_FILHOS, calculadas a partir de clientes únicos
filhos = clientes["NUMERO_FILHOS"]

media = filhos.mean()
mediana = filhos.median()
desvio_padrao = filhos.std()
moda = filhos.mode()[0]        # mode() pode retornar mais de um valor se houver empate, pega o primeiro
maximo = filhos.max()
minimo = filhos.min()
contagem = filhos.count()
q1 = filhos.quantile(0.25)
q2 = filhos.quantile(0.50)
q3 = filhos.quantile(0.75)

print(f"Média: {media:.2f}")
print(f"Mediana: {mediana}")
print(f"Desvio Padrão: {desvio_padrao:.2f}")
print(f"Moda: {moda}")
print(f"Máximo: {maximo}")
print(f"Mínimo: {minimo}")
print(f"Contagem: {contagem}")
print(f"1º Quartil (25%): {q1}")
print(f"2º Quartil (50%): {q2}")
print(f"3º Quartil (75%): {q3}")

Média: 1.14
Mediana: 0.0
Desvio Padrão: 1.41
Moda: 0
Máximo: 4
Mínimo: 0
Contagem: 1000
1º Quartil (25%): 0.0
2º Quartil (50%): 0.0
3º Quartil (75%): 2.0


Essas estatísticas poderiam ter sido feitas utilizando apenas o .describe(), mas para fins de melhor
visualização e para incluir a moda (que o .describe() não calcula), preferi separar valor por valor com print.